In [0]:
%run "./00_setup_and_raw_data_ingestion"

In [0]:
from pyspark.sql import functions as F

resolved_statuses = ["Fully Paid", "Charged Off", "Default"]

df_lc_resolved = df_lc.filter(F.col("loan_status").isin(resolved_statuses))

df_lc_resolved = df_lc_resolved.withColumn(
    "target",
    F.when(F.col("loan_status") == "Fully Paid", 0).otherwise(1)
)

print(f"Rows before filtering: {df_lc.count()}")
print(f"Rows after filtering to resolved loans: {df_lc_resolved.count()}")
df_lc_resolved.groupBy("target").count().show()

In [0]:
leakage_cols = [
    "out_prncp_inv", "total_pymnt_inv", "total_rec_prncp",
    "total_rec_int", "total_rec_late_fee", "collection_recovery_fee",
    "last_pymnt_amnt", "debt_settlement_flag", "loan_status"
]

df_lc_resolved = df_lc_resolved.drop(*[c for c in leakage_cols if c in df_lc_resolved.columns])

print(f"Columns remaining: {len(df_lc_resolved.columns)}")

In [0]:
# ============================================================
# 3. Missing value ratio per column — same threshold as the
#    local pandas exploration (>50% missing → drop candidate)
# ============================================================
from pyspark.sql import functions as F

total_rows = df_lc_resolved.count()

missing_ratios = df_lc_resolved.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_rows).alias(c)
    for c in df_lc_resolved.columns
])

missing_pd = missing_ratios.toPandas().T.rename(columns={0: "missing_ratio"})
high_missing_cols = missing_pd[missing_pd["missing_ratio"] > 0.5].index.tolist()

print(f"Columns with >50% missing: {len(high_missing_cols)}")
print(high_missing_cols)

In [0]:
df_lc_resolved = df_lc_resolved.drop(*high_missing_cols)
print(f"Columns remaining: {len(df_lc_resolved.columns)}")

In [0]:
# ============================================================
# 4. Near-constant columns — same criterion as the local
#    pandas exploration
# ============================================================
from pyspark.sql import functions as F

near_constant_threshold = 0.99  # if one value covers >99% of non-null rows
near_constant_cols = []

for c in df_lc_resolved.columns:
    top_value_ratio = (
        df_lc_resolved.groupBy(c).count()
        .orderBy(F.desc("count"))
        .limit(1)
        .withColumn("ratio", F.col("count") / total_rows)
        .select("ratio")
        .collect()[0]["ratio"]
    )
    if top_value_ratio > near_constant_threshold:
        near_constant_cols.append(c)

print(f"Near-constant columns: {len(near_constant_cols)}")
print(near_constant_cols)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 4. Near-constant columns — single-pass version (much faster
#    than one groupBy per column)
# ============================================================
near_constant_threshold = 0.99
near_constant_cols = []

# For each column, get the count of its most frequent value in
# one aggregation pass per column, but avoid the expensive
# orderBy + limit + collect pattern — use a cheaper max() instead
for c in df_lc_resolved.columns:
    counts = df_lc_resolved.groupBy(c).count()
    max_count = counts.agg(F.max("count")).collect()[0][0]
    ratio = max_count / total_rows
    if ratio > near_constant_threshold:
        near_constant_cols.append(c)

print(f"Near-constant columns: {len(near_constant_cols)}")
print(near_constant_cols)

In [0]:
# Much faster: compute on a 10% sample instead of the full dataset
df_sample = df_lc_resolved.sample(fraction=0.1, seed=42)
sample_total = df_sample.count()

near_constant_cols = []
for c in df_sample.columns:
    max_count = df_sample.groupBy(c).count().agg(F.max("count")).collect()[0][0]
    if (max_count / sample_total) > near_constant_threshold:
        near_constant_cols.append(c)

print(f"Near-constant columns (estimated from 10% sample): {len(near_constant_cols)}")
print(near_constant_cols)

In [0]:
# ============================================================
# Stop the previous cell first, then run this
# ============================================================
df_sample = df_lc_resolved.sample(fraction=0.1, seed=42)

# .toPandas() forces ONE single materialization — after this,
# pdf_sample lives in local memory, no more repeated Spark
# recomputation from the raw file for each column
pdf_sample = df_sample.toPandas()
print(f"Sample shape: {pdf_sample.shape}")

In [0]:
df_sample = df_lc_resolved.sample(fraction=0.01, seed=42)  # 1% invece di 10%
pdf_sample = df_sample.toPandas()
print(f"Sample shape: {pdf_sample.shape}")

In [0]:
df_lc_resolved.explain()

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Near-constant detection via freqItems — a single distributed
# pass over the whole dataset (approximate algorithm), instead
# of one groupBy per column. Avoids both the 79x re-scan problem
# and the Parquet write restriction entirely.
# ============================================================
freq_result = df_lc_resolved.stat.freqItems(df_lc_resolved.columns, support=0.99)

# freq_result has one row; each column is named "<original_col>_freqItems"
# and contains an array of values that appear in >99% of rows (if any)
row = freq_result.collect()[0]

near_constant_cols = [
    c for c in df_lc_resolved.columns
    if len(row[f"{c}_freqItems"]) > 0
]

print(f"Near-constant columns: {len(near_constant_cols)}")
print(near_constant_cols)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Step 1 — approx_count_distinct for ALL columns in ONE pass.
# Cheap and immediately rules out high-cardinality columns
# (loan_amnt, int_rate, etc.) without needing a per-column groupBy.
# ============================================================
distinct_counts = df_lc_resolved.agg(
    *[F.approx_count_distinct(c).alias(c) for c in df_lc_resolved.columns]
).collect()[0]

# Candidates: columns with very few distinct values relative to
# row count — anything with many distinct values cannot be near-constant
candidate_cols = [
    c for c in df_lc_resolved.columns
    if distinct_counts[c] <= 50
]

print(f"Candidates for near-constant check: {len(candidate_cols)}")
print(candidate_cols)

In [0]:
# ============================================================
# Step 2 — exact top-value ratio, but only on the small
# candidate list from step 1 (fast, since few columns remain)
# ============================================================
near_constant_cols = []

for c in candidate_cols:
    max_count = df_lc_resolved.groupBy(c).count().agg(F.max("count")).collect()[0][0]
    if (max_count / total_rows) > 0.99:
        near_constant_cols.append(c)

print(f"Near-constant columns: {len(near_constant_cols)}")
print(near_constant_cols)

In [0]:
df_lc_resolved = df_lc_resolved.drop(*near_constant_cols)
print(f"Columns remaining: {len(df_lc_resolved.columns)}")

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd

# ============================================================
# Correlation matrix — single distributed pass via pyspark.ml,
# instead of one query per column pair. handleInvalid="skip"
# drops rows with any null among the numeric columns for this
# calculation only (doesn't affect df_lc_resolved itself).
# ============================================================
numeric_cols = [
    c for c, dtype in df_lc_resolved.dtypes
    if dtype in ("int", "double", "bigint", "float") and c != "target"
]

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features", handleInvalid="skip")
df_vector = assembler.transform(df_lc_resolved).select("features")

corr_matrix = Correlation.corr(df_vector, "features").head()[0].toArray()
corr_pd = pd.DataFrame(corr_matrix, index=numeric_cols, columns=numeric_cols)

print(f"Correlation matrix shape: {corr_pd.shape}")

In [0]:
# ============================================================
# Extract pairs above threshold — same logic as the local
# pandas exploration
# ============================================================
import numpy as np

threshold = 0.85
pairs = []
for i in range(len(corr_pd.columns)):
    for j in range(i + 1, len(corr_pd.columns)):
        val = corr_pd.iloc[i, j]
        if abs(val) > threshold:
            pairs.append((corr_pd.columns[i], corr_pd.columns[j], round(val, 3)))

pairs_df = pd.DataFrame(pairs, columns=["col_1", "col_2", "corr"]).sort_values("corr", ascending=False)
print(pairs_df)

In [0]:
drop_correlation = [
    "funded_amnt", "funded_amnt_inv",       # duplicati di loan_amnt
    "num_rev_tl_bal_gt_0",                   # duplicato di num_actv_rev_tl
    "total_il_high_credit_limit",            # ridondante con total_bal_ex_mort
    "bc_open_to_buy",                        # ridondante con total_bc_limit
]

df_lc_resolved = df_lc_resolved.drop(*[c for c in drop_correlation if c in df_lc_resolved.columns])
print(f"Columns remaining: {len(df_lc_resolved.columns)}")

In [0]:
# ============================================================
# Family-based selection — same reasoning as the local pandas
# exploration (docs/feature_engineering.md, feature family
# grouping). Pure column drops, no computation — should be fast.
# ============================================================

# Family A: time-since-event — keep mo_sin_old_rev_tl_op, mths_since_recent_inq
drop_family_a = ["mo_sin_old_il_acct", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mths_since_recent_bc"]

# Family B: account counts by type — keep open_acc, num_actv_rev_tl
drop_family_b = ["total_acc", "num_actv_bc_tl", "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl", "num_rev_accts"]

# Family C: delinquency/derogatory counts — keep delinq_2yrs, num_tl_90g_dpd_24m, pub_rec_bankruptcies, pct_tl_nvr_dlq
drop_family_c = ["num_accts_ever_120_pd", "num_tl_120dpd_2m", "num_tl_30dpd", "pub_rec", "tax_liens"]

# Family D: recently-opened accounts — keep num_tl_op_past_12m, acc_open_past_24mths
drop_family_d = []  # already reduced upstream (most variants were in the >50% missing drop)

# Family E: utilization/balance ratios — keep revol_util, all_util, avg_cur_bal, total_bc_limit
drop_family_e = ["percent_bc_gt_75", "total_rev_hi_lim"]

# Family F: inquiries — keep inq_last_6mths
drop_family_f = []  # inq_fi, inq_last_12m already gone (>50% missing)

# Non-credit-history admin fields, low expected value
drop_admin = ["zip_code", "last_credit_pull_d", "policy_code", "pymnt_plan", "initial_list_status"]

columns_to_drop = drop_family_a + drop_family_b + drop_family_c + drop_family_d + drop_family_e + drop_family_f + drop_admin

df_lc_resolved = df_lc_resolved.drop(*[c for c in columns_to_drop if c in df_lc_resolved.columns])
print(f"Columns remaining: {len(df_lc_resolved.columns)}")
print(sorted(df_lc_resolved.columns))

In [0]:
additional_leakage = ["out_prncp", "total_pymnt", "recoveries", "last_pymnt_d"]

df_lc_resolved = df_lc_resolved.drop(*[c for c in additional_leakage if c in df_lc_resolved.columns])
print(f"Columns remaining: {len(df_lc_resolved.columns)}")